# 19.4 Concept-based explanations (TCAV)

TCAV asks whether movement along a human concept direction tends to increase a class score.

A positive concept sensitivity is evidence about a direction in activation space, not proof of causality.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(19)


def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def fit_ladder_classifier(X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr_s = scaler.fit_transform(x_tr)
    x_te_s = scaler.transform(x_te)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr_s, y_tr)
    return clf, scaler, x_tr_s, x_te_s, y_tr, y_te


def score_for_class(model, X, class_index=None):
    proba = model.predict_proba(X)
    if class_index is None:
        class_index = int(np.argmax(proba[0]))
    return proba[:, class_index]


def finite_gradient(fn, x, eps=1e-4):
    grad = np.zeros_like(x, dtype=float)
    for j in range(len(x)):
        plus = x.copy()
        minus = x.copy()
        plus[j] = plus[j] + eps
        minus[j] = minus[j] - eps
        grad[j] = (fn(plus) - fn(minus)) / (2.0 * eps)
    return grad


def rank_corr(a, b):
    ar = pd.Series(a).rank(method="average").to_numpy()
    br = pd.Series(b).rank(method="average").to_numpy()
    if np.std(ar) == 0.0 or np.std(br) == 0.0:
        return 0.0
    return float(np.corrcoef(ar, br)[0, 1])


def top_k_mask(values, k):
    order = np.argsort(-np.abs(values))
    mask = np.zeros(len(values), dtype=bool)
    mask[order[:k]] = True
    return mask


## The concept, built once (D1)

TCAV scores a class along a concept vector: $S_{C,k}(x)=
abla h_k(x)^	op v_C$. The concept direction comes from concept examples versus controls, then the class gradient supplies sensitivity.

In [ ]:

def concept_direction(concept_examples, random_examples):
    raw = np.mean(concept_examples, axis=0) - np.mean(random_examples, axis=0)
    norm = np.linalg.norm(raw)
    if norm == 0.0:
        return raw
    return raw / norm


def tcav_score(activations, concept_examples, random_examples, class_grad):
    v_c = concept_direction(concept_examples, random_examples)
    components = class_grad * v_c
    scores = activations @ v_c
    directional_score = float(class_grad @ v_c)
    return directional_score, components, scores, v_c

v_raw = np.array([1.0, 1.0, -1.0])
v_unit = v_raw / np.linalg.norm(v_raw)
concept_examples_toy = np.array([v_unit, v_unit + 0.1])
random_examples_toy = np.zeros((2, 3))
class_grad_toy = np.array([0.7, 0.4, 0.2]) / v_unit
activations_toy = np.eye(3)
score_toy, components_toy, scores_toy, v_c_toy = tcav_score(activations_toy, concept_examples_toy, random_examples_toy, class_grad_toy)
abs_toy = float(np.abs(components_toy).sum())
share_toy = float(np.max(np.abs(components_toy)) / abs_toy)
guarded_toy = score_toy + 0.6 * abs_toy
contrast_toy = score_toy - components_toy[2]

assert np.allclose(components_toy, [0.7, 0.4, -0.2])
assert np.isclose(score_toy, 0.9)
assert np.isclose(abs_toy, 1.3)
assert np.isclose(round(share_toy, 3), 0.538)
assert np.isclose(guarded_toy, 1.68)
assert np.isclose(contrast_toy - score_toy, 0.2)

print("concept direction", np.round(v_c_toy, 3))
print("TCAV components", components_toy)
print("TCAV score", score_toy)


For the ladder, standardized features act as a simple activation layer. A concept is the direction from low predicted-score examples to high predicted-score examples, and faithfulness is measured by ablation drop along that direction plus sign stability.

In [ ]:

def ladder_tcav(model, x_tr, x_te):
    proba = model.predict_proba(x_tr)
    class_index = int(np.argmax(model.predict_proba(x_te[0].reshape(1, -1))[0]))
    class_scores = proba[:, class_index]
    high = x_tr[class_scores >= np.quantile(class_scores, 0.8)]
    low = x_tr[class_scores <= np.quantile(class_scores, 0.2)]
    v_c = concept_direction(high, low)

    def local_score(z):
        return float(score_for_class(model, z.reshape(1, -1), class_index=class_index)[0])

    grad = finite_gradient(local_score, x_te[0])
    sensitivity = float(grad @ v_c)
    edited = x_te[0] - 0.5 * v_c
    original_score = local_score(x_te[0])
    edited_score = local_score(edited)
    ablation_drop = original_score - edited_score
    signs = []
    for row in x_te[: min(30, len(x_te))]:
        row_grad = finite_gradient(local_score, row)
        signs.append(np.sign(row_grad @ v_c))
    sign_stability = float(np.mean(np.array(signs) == np.sign(sensitivity)))
    return sensitivity, ablation_drop, sign_stability, v_c, grad


## The dataset ladder

We use the shared F15 classification ladder: a hand linear toy, clean blobs, a nonlinear moons stress test, real Wine, and real Breast Cancer. The same logistic base model and the same metric code run unchanged across rungs.

In [ ]:

rungs = clf_ladder()

for index, item in enumerate(rungs, start=1):
    name, X, y = item
    counts = dict(zip(*np.unique(y, return_counts=True)))
    print(index, name)
    print("shape", X.shape)
    print("class counts", counts)
    print("sample", np.round(X[:3, : min(5, X.shape[1])], 3))


## Run the SAME method across D1-D5

The metric is concept faithfulness: sign stability, with ablation drop as the reliability check.

In [ ]:

tcav_rows = []
tcav_artifacts = []

for rung, item in enumerate(rungs, start=1):
    name, X, y = item
    model, scaler, x_tr, x_te, y_tr, y_te = fit_ladder_classifier(X, y)
    sensitivity, ablation_drop, sign_stability, v_c, grad = ladder_tcav(model, x_tr, x_te)
    tcav_rows.append({"rung": rung, "name": name, "sensitivity": sensitivity, "ablation_drop": ablation_drop, "sign_stability": sign_stability})
    tcav_artifacts.append((name, v_c, grad))

tcav_table = pd.DataFrame(tcav_rows)
print(tcav_table.to_string(index=False))


## Results visualization

Panels show concept vectors and class gradients. The curve tracks sign stability as stress rises.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.ravel()

for ax, artifact in zip(axes[:5], tcav_artifacts):
    name, v_c, grad = artifact
    keep = min(10, len(v_c))
    ax.bar(np.arange(keep) - 0.2, v_c[:keep], width=0.4, label="concept")
    ax.bar(np.arange(keep) + 0.2, grad[:keep], width=0.4, label="gradient")
    ax.axhline(0.0, color="black", linewidth=0.8)
    ax.set_title(name[:26])
    ax.legend(fontsize=8)

axes[5].plot(tcav_table["rung"], tcav_table["sign_stability"], marker="o")
axes[5].set_title("sensitivity stability vs stress")
axes[5].set_xlabel("rung")
axes[5].set_ylabel("sign stability")
axes[5].set_ylim(0.0, 1.05)
plt.tight_layout()


## Pitfall on the hardest rung

Pitfall: concept examples that overlap the target class can capture class identity instead of the concept. The fix compares held-out concept negatives and layer/feature projections.

In [ ]:

name, X, y = rungs[-1]
model, scaler, x_tr, x_te, y_tr, y_te = fit_ladder_classifier(X, y)
class_index = int(np.argmax(model.predict_proba(x_te[0].reshape(1, -1))[0]))
proba = model.predict_proba(x_tr)[:, class_index]
leaky_concept = x_tr[y_tr == class_index]
controls = x_tr[y_tr != class_index]
leaky_v = concept_direction(leaky_concept, controls)
clean_high = x_tr[proba >= np.quantile(proba, 0.8)]
clean_low = x_tr[proba <= np.quantile(proba, 0.2)]
clean_v = concept_direction(clean_high, clean_low)
leak_overlap = float(np.mean(y_tr[y_tr == class_index] == class_index))
cosine = float(leaky_v @ clean_v)

print("leaky concept class overlap", leak_overlap)
print("cosine between leaky and held-out concept directions", cosine)
print("use held-out negatives and compare directions before trusting TCAV")


## Evaluate it + Practice

- Compare the reported concept sign stability with a no-skill baseline such as shuffled features, shuffled groups, or random rankings.
- Sanity check signs, denominators, and the background/reference point before trusting a pretty plot.
- Ablation: define concepts from target-class examples only and compare the direction with held-out controls
- Failure signals: unstable ranks under small perturbations, a metric that improves while accuracy collapses, or explanations that change when the seed changes.
- For gap topics, especially influence functions, keep the lesson numbers visible and treat the notebook as an audit scaffold until the lesson body grows more examples.

Practice 1: Change the trust knob or kernel width and predict which metric should move before running it.

Practice 2: Swap the D5 background/reference set and explain which conclusion is no longer stable.

Practice 3: Turn the pitfall back on, then write one sentence explaining why the fixed version is safer.